# Data Preparation: Part 4 - Reporter Function

## 1. Import Packages

In [1]:
import pandas as pd
import re
from feature_engine.imputation import CategoricalImputer

## 2. Import Raw Data

In [2]:
full = pd.read_csv('Raw Input Files/asrs_full.csv')

In [3]:
full

,acn,event_date,anomaly_code,location_id,aircraft_type,reporter_function,flight_phase,narrative_text,synopsis,word_count,hedge_score,passive_voice_ratio,causal_connective_count,specificity_score,named_entity_count,type_token_ratio,recurrence_flag,days_to_recurrence,is_right_censored,label_method
0,1507557,2018-01-01,ATC Issue All Types; Deviation / Discrepancy -...,C90.TRACON,A330,Approach,Landing,Aircraft X was assigned a runway approximately...,MLI Approach Controller reported ATC denied th...,143,0.000000,0.5556,2,0.00,7,0.5833,1,181.0,0,exact_anomaly_code_same_location_12mo
1,1513720,2018-01-01,Aircraft Equipment Problem Less Severe,ZZZ.Airport,EMB ERJ 170/175 ER/LR,Captain; Pilot Not Flying,Cruise,Pack 2 was MELed. My FO (First Officer) and I ...,ERJ-175 Captain reported diverting after exper...,318,0.003145,0.2105,3,0.75,19,0.4969,0,NaN,0,exact_anomaly_code_same_location_12mo
2,1513718,2018-01-01,Flight Deck / Cabin / Aircraft Event Illness /...,ZZZ.ARTCC,A321,Pilot Not Flying,Cruise,[A passenger] was reported as being ill and un...,A321 flight crew member reported difficulty re...,166,0.000000,0.1111,0,0.00,13,0.5689,1,120.0,0,exact_anomaly_code_same_location_12mo
3,1513706,2018-01-01,Aircraft Equipment Problem Critical,ZZZ.Airport,Citation V/Ultra/Encore (C560),First Officer; Pilot Flying,Final Approach,While on approach at 1;500ft; we asked the tow...,CE-560 First Officer reported that while on ap...,268,0.000000,0.0556,1,0.50,11,0.5221,0,NaN,0,exact_anomaly_code_same_location_12mo
4,1513663,2018-01-01,Aircraft Equipment Problem Less Severe; Deviat...,RCTP.Airport,B747-400,Pilot Flying; Captain,NaN,Our aircraft had just finished a 'heavy' maint...,B747-400 Captain reported a loss of hydraulic ...,190,0.000000,0.0667,0,0.25,5,0.5812,0,NaN,0,exact_anomaly_code_same_location_12mo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33718,2192919,2024-12-01,Aircraft Equipment Problem Critical; Deviation...,ZZZ.Airport,Skyhawk 172/Cutlass 172,Instructor,Climb,This occurred shortly after departing from ZZZ...,C172 Flight Instructor reported cockpit smoke ...,258,0.000000,0.2308,0,0.25,19,0.6151,0,NaN,1,exact_anomaly_code_same_location_12mo
33719,2192908,2024-12-01,Deviation - Altitude Overshoot; Deviation / Di...,CRQ.Airport,Any Unknown or Unlisted Aircraft Manufacturer,Single Pilot; Pilot Flying,Final Approach,I departed from ZZZ with a filed IFR flight pl...,GA pilot reported they descended below the Min...,579,0.001727,0.2800,4,0.50,33,0.4220,0,NaN,1,exact_anomaly_code_same_location_12mo
33720,2192884,2024-12-01,Conflict NMAC; Deviation / Discrepancy - Proce...,ZZZ.Airport,Skyhawk 172/Cutlass 172,Instructor,Final Approach,Practice approach RNAV XX into ZZZ. I am the C...,C172 flight instructor reported a near mid-air...,63,0.000000,0.0000,0,0.25,8,0.7969,0,NaN,1,exact_anomaly_code_same_location_12mo
33721,2192873,2024-12-01,Deviation / Discrepancy - Procedural Published...,ZZZ.Airport,A319,Ramp,Taxi,My shift started at XA:00 AM and I was assigne...,Tow team lead reported after parking aircraft ...,1930,0.005181,0.3210,5,0.75,49,0.2664,0,NaN,1,exact_anomaly_code_same_location_12mo


## 3. Clean Data

### 3.1. Prelim Data Cleaning

In [4]:
# Replace 'reporter_function' with all uppercase
df = full.copy()
df['reporter_function'] = df['reporter_function'].str.upper()

### 3.2. Treat Missing Values

In [5]:
# Identify variables with any missing values
data_miss = df.columns[df.isnull().any()].tolist()

In [6]:
# List missing categorical columns
# cat = data_raw.columns[data_raw.dtypes == 'object'].tolist()
# cat = df[data_miss].columns[df[data_miss].dtypes == 'object'].tolist()
# cat

In [7]:
# Replace categorical columns
imp = CategoricalImputer(imputation_method = 'missing', 
                         fill_value = 'OTHER / UNKNOWN',
                         variables = 'reporter_function')

df = imp.fit_transform(df)

In [8]:
with pd.option_context('display.max_rows', None):
    display(df[['reporter_function']].value_counts())

reporter_function                                                          
CAPTAIN; PILOT FLYING                                                          4760
PILOT FLYING; CAPTAIN                                                          2427
CAPTAIN; PILOT NOT FLYING                                                      2345
PILOT FLYING; SINGLE PILOT                                                     2208
FIRST OFFICER; PILOT NOT FLYING                                                1845
FIRST OFFICER; PILOT FLYING                                                    1510
CAPTAIN                                                                        1454
PILOT NOT FLYING; CAPTAIN                                                      1192
PILOT FLYING                                                                   1189
SINGLE PILOT; PILOT FLYING                                                     1069
PILOT NOT FLYING; FIRST OFFICER                                                1059


### 3.3. Identify Unique Categories

In [9]:
functions = df.reporter_function

In [10]:
disagg_functions = []
for phrase in functions:
    disagg_functions.extend([s.strip() for s in re.split(r';', phrase)])

In [11]:
disagg_functions = pd.Series(disagg_functions)

In [12]:
# Counts
value_counts = disagg_functions.value_counts().reset_index()
value_counts.columns = ["Functions", "Count"]

In [13]:
with pd.option_context('display.max_rows', None):
    display(value_counts)

,Functions,Count
0,PILOT FLYING,15236
1,CAPTAIN,13018
2,PILOT NOT FLYING,8235
3,FIRST OFFICER,5845
4,SINGLE PILOT,4853
5,INSTRUCTOR,2716
6,APPROACH,951
7,ENROUTE,898
8,FLIGHT ATTENDANT (ON DUTY),789
9,OTHER / UNKNOWN,681


### 3.4. Consolidate / Map Categories

Mapped per: https://akama.arc.nasa.gov/ASRSDBOnline/CodingForm.pdf

In [14]:
# JW NOTE: See "... Compilation/Int/Potential ASRS mapping.xlsx" (will need to download to view)
    # FYI I added a new category for instructors/examiners -- lmk if you think we should categorize these differently
    # Added some potential mapping with the assumption we're including all categories once delimited
    # Also split up the dictionary below into a definition + updates bc it was getting a little long

In [15]:
df_map = df.copy()

In [16]:
df_map

,acn,event_date,anomaly_code,location_id,aircraft_type,reporter_function,flight_phase,narrative_text,synopsis,word_count,hedge_score,passive_voice_ratio,causal_connective_count,specificity_score,named_entity_count,type_token_ratio,recurrence_flag,days_to_recurrence,is_right_censored,label_method
0,1507557,2018-01-01,ATC Issue All Types; Deviation / Discrepancy -...,C90.TRACON,A330,APPROACH,Landing,Aircraft X was assigned a runway approximately...,MLI Approach Controller reported ATC denied th...,143,0.000000,0.5556,2,0.00,7,0.5833,1,181.0,0,exact_anomaly_code_same_location_12mo
1,1513720,2018-01-01,Aircraft Equipment Problem Less Severe,ZZZ.Airport,EMB ERJ 170/175 ER/LR,CAPTAIN; PILOT NOT FLYING,Cruise,Pack 2 was MELed. My FO (First Officer) and I ...,ERJ-175 Captain reported diverting after exper...,318,0.003145,0.2105,3,0.75,19,0.4969,0,NaN,0,exact_anomaly_code_same_location_12mo
2,1513718,2018-01-01,Flight Deck / Cabin / Aircraft Event Illness /...,ZZZ.ARTCC,A321,PILOT NOT FLYING,Cruise,[A passenger] was reported as being ill and un...,A321 flight crew member reported difficulty re...,166,0.000000,0.1111,0,0.00,13,0.5689,1,120.0,0,exact_anomaly_code_same_location_12mo
3,1513706,2018-01-01,Aircraft Equipment Problem Critical,ZZZ.Airport,Citation V/Ultra/Encore (C560),FIRST OFFICER; PILOT FLYING,Final Approach,While on approach at 1;500ft; we asked the tow...,CE-560 First Officer reported that while on ap...,268,0.000000,0.0556,1,0.50,11,0.5221,0,NaN,0,exact_anomaly_code_same_location_12mo
4,1513663,2018-01-01,Aircraft Equipment Problem Less Severe; Deviat...,RCTP.Airport,B747-400,PILOT FLYING; CAPTAIN,NaN,Our aircraft had just finished a 'heavy' maint...,B747-400 Captain reported a loss of hydraulic ...,190,0.000000,0.0667,0,0.25,5,0.5812,0,NaN,0,exact_anomaly_code_same_location_12mo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33718,2192919,2024-12-01,Aircraft Equipment Problem Critical; Deviation...,ZZZ.Airport,Skyhawk 172/Cutlass 172,INSTRUCTOR,Climb,This occurred shortly after departing from ZZZ...,C172 Flight Instructor reported cockpit smoke ...,258,0.000000,0.2308,0,0.25,19,0.6151,0,NaN,1,exact_anomaly_code_same_location_12mo
33719,2192908,2024-12-01,Deviation - Altitude Overshoot; Deviation / Di...,CRQ.Airport,Any Unknown or Unlisted Aircraft Manufacturer,SINGLE PILOT; PILOT FLYING,Final Approach,I departed from ZZZ with a filed IFR flight pl...,GA pilot reported they descended below the Min...,579,0.001727,0.2800,4,0.50,33,0.4220,0,NaN,1,exact_anomaly_code_same_location_12mo
33720,2192884,2024-12-01,Conflict NMAC; Deviation / Discrepancy - Proce...,ZZZ.Airport,Skyhawk 172/Cutlass 172,INSTRUCTOR,Final Approach,Practice approach RNAV XX into ZZZ. I am the C...,C172 flight instructor reported a near mid-air...,63,0.000000,0.0000,0,0.25,8,0.7969,0,NaN,1,exact_anomaly_code_same_location_12mo
33721,2192873,2024-12-01,Deviation / Discrepancy - Procedural Published...,ZZZ.Airport,A319,RAMP,Taxi,My shift started at XA:00 AM and I was assigne...,Tow team lead reported after parking aircraft ...,1930,0.005181,0.3210,5,0.75,49,0.2664,0,NaN,1,exact_anomaly_code_same_location_12mo


In [17]:
df_map['reporter_function_agg'] = "; "+df_map['reporter_function']

In [18]:
# agg_pilot = ['CAPTAIN', 'FIRST OFFICER', 'PILOT FLYING', 'PILOT NOT FLYING', 'SINGLE PILOT', 'RELIEF PILOT', 'CHECK PILOT', 'STUDENT PILOT', 'DEADHEADING PILOT', 'TRAINEE', 'FLIGHT ENGINEER / SECOND OFFICER', 'DRONE OPERATOR', 'REMOTE PIC (UAS)', 'PERSON MANIPULATING CONTROLS (UAS)', 'VISUAL OBSERVER (UAS)', 'REMOTE PILOT']
# df_map.loc[df_map['reporter_function_agg'].isin(agg_pilot), 'reporter_function_agg'] = "PILOT"

In [19]:
# JW NOTE: There's probabaly a cleaner way to do this with the dictionary but in case not, here's a brute forced version with loops:

In [20]:
# Instruction / Exam
agg_INSTRUCTION_EXAM = [
    '; CFI-GFRONT;COMM-GREAR', 
    '; CFII WAS PRESENT', 
    '; DESIGNATED PILOT EXAMINER', 
    '; DUAL FLIGHT INSTRUCTION', 
    '; DUAL INSTRUCTION FLIGHT', 
    '; FLYING WITH INSTRUCTOR', 
    '; INSTRUMENT STUDENT', 
    '; LINE CHECK AIRMAN/CAPTAIN', 
    '; LINEMAN', 
    '; SAFETY PILOT', 
    '; STUDENTS INSTRUCTOR', 
    '; SUPERVISOR OF FLYING', 
    '; WITH DPE',
    '; CFI', 
    '; DPE', 
    '; DUAL FLIGHT', 
    '; LINE CHECK AIRMAN', 
]
for x in agg_INSTRUCTION_EXAM:
    df_map['reporter_function_agg'] = df_map['reporter_function_agg'].str.replace(x, repl = "; INSTRUCTION_EXAM")

In [21]:
# Pilot
agg_PILOT = [
    '; CHECK PILOT', 
    '; CIVL AIR PTRL CHECK PILOT', 
    '; COMMERCIAL STUDENT PILOT', 
    '; CPL STUDENT', 
    '; DEADHEADING PILOT', 
    '; DRONE OPERATOR', 
    '; FIRST OFFICER', 
    '; HELIPORT MANAGER & PILOT', 
    '; INSTRUCTOR', 
    '; MULTI-ENGINE STUDENT', 
    '; PARAMOTOR PILOT', 
    '; PERSON MANIPULATING CONTROLS (UAS)', 
    '; PIC DURING TIME BUILD', 
    '; PIC NOT FLYING', 
    '; PIC ON A CHECKRIDE', 
    '; PILOT FLYING', 
    '; PILOT IN FOGGLES', 
    '; PILOT NOT FLYING', 
    '; PILOT RECIVING TYPE TRN', 
    '; PRIV. PILOT W/ INSTRUCTOR', 
    '; PRIVATE PILOT', 
    '; RELIEF PILOT', 
    '; REMOTE PIC (UAS)', 
    '; REMOTE PILOT', 
    '; SINGLE PILOT', 
    '; SOLO STUDENT', 
    '; STUDENT PILOT & INSTR', 
    '; STUDENT PILOT W/ CFI', 
    '; STUDENT SOLO PILOT', 
    '; TRAINEE', 
    '; UAS SUPERVISOR', 
    '; VISUAL OBSERVER (UAS)', 
    '; CAPTAIN', 
    '; COMMERCIAL STUDENT', 
    '; PIC', 
    '; STUDENT PILOT',
    '; STUDENT'
]
for x in agg_PILOT:
    df_map['reporter_function_agg'] = df_map['reporter_function_agg'].str.replace(x, repl = "; PILOT")

In [22]:
# ATC
agg_ATC = [
    '; APPROACH', 
    '; ATC', 
    '; COORDINATOR', 
    '; DEPARTURE', 
    '; DISPATCHER', 
    '; ENROUTE', 
    '; FLIGHT DATA / CLEARANCE DELIVERY', 
    '; HANDOFF / ASSIST', 
    '; SUPERVISOR / CIC', 
    '; TRAFFIC MANAGEMENT', 
    '; UNICOM OPERATIONS'
]
for x in agg_ATC:
    df_map['reporter_function_agg'] = df_map['reporter_function_agg'].str.replace(x, repl = "; ATC")

In [23]:
# Ground
agg_GROUND = [
    '; AIRPORT MANAGER', 
    '; AIRPORT PERSONNEL', 
    '; FBO MANAGER', 
    '; FBO PERSONNEL', 
    '; GATE AGENT / CSR', 
    '; JET BRIDGE OPERATOR', 
    '; LOAD CCONTROLLER', 
    '; LOAD PLANNER', 
    '; RAMP', 
    '; VEHICLE DRIVER',
    '; GROUND'
]
for x in agg_GROUND:
    df_map['reporter_function_agg'] = df_map['reporter_function_agg'].str.replace(x, repl = "; GROUND")

In [24]:
# Maintenance / Safety
agg_MAINT_SAFETY = [
    '; AVIATION SAFETY OFFICER', 
    '; FLIGHT ENGINEER / SECOND OFFICER', 
    '; FLIGHT SAFETY OFFICER', 
    '; FSDO POI', 
    '; LEAD TECHNICIAN', 
    '; MAINTENANCE MANAGER', 
    '; MANAGER - REGULATORY COMPLIANCE', 
    '; PARTS / STORES PERSONNEL', 
    '; QUALITY ASSURANCE / AUDIT', 
    '; SAFETY ANALYST', 
    '; SAFETY ANANLYST', 
    '; SAFETY INVESTIGATOR', 
    '; SAFETY MANAGER', 
    '; SAFETY OFFICER', 
    '; SMS ANALYST', 
    '; INSPECTOR', 
    '; MANAGER',
    '; SAFETY OFFICE', 
    '; TECHNICIAN'
]
for x in agg_MAINT_SAFETY:
    df_map['reporter_function_agg'] = df_map['reporter_function_agg'].str.replace(x, repl = "; MAINT_SAFETY")

In [25]:
# Other Crew
agg_OTH_CREW = [
    '; ARMY H-60 CREW', 
    '; FLIGHT ATTENDANT (ON DUTY)', 
    '; FLIGHT ATTENDANT IN CHARGE', 
    '; FLIGHT SERVICE', 
    '; MEDICAL CREW', 
    '; OFF DUTY'
]
for x in agg_OTH_CREW:
    df_map['reporter_function_agg'] = df_map['reporter_function_agg'].str.replace(x, repl = "; OTH_CREW")

In [26]:
# Other / Unknown
agg_OTH_UNKN = [
    '; CITIZEN', 
    '; COMPANY CEO', 
    '; LEFT SEAT DURING SIM IFR', 
    '; LOCAL', 
    '; NO PASSENGERS', 
    '; OBSERVER/DEADHEADER', 
    '; OCEANIC', 
    '; OTHER / UNKNOWN', 
    '; PASSENGER', 
    '; SKYDIVER', 
    '; OBSERVER', 
    '; OTHER', 
    '; UNKNOWN'
]
for x in agg_OTH_UNKN:
    df_map['reporter_function_agg'] = df_map['reporter_function_agg'].str.replace(x, repl = "; OTH_UNKN")

In [27]:
with pd.option_context('display.max_rows', None):
    display(df_map['reporter_function_agg'].value_counts())

# with pd.option_context('display.max_rows', None):
#     display(df_map['reporter_function_agg'].head(5000))

reporter_function_agg
; PILOT; PILOT                                          21389
; PILOT                                                  5832
; ATC                                                    1985
; OTH_UNKN                                               1128
; OTH_CREW                                                863
; MAINT_SAFETY                                            692
; PILOT; PILOT; PILOT                                     587
; GROUND                                                  363
; ATC; ATC                                                149
; ATC; PILOT                                               80
; OTH_CREW; OTH_CREW                                       72
; MAINT_SAFETY; MAINT_SAFETY                               52
; PILOT; ATC                                               48
; ATC; PILOT; PILOT                                        39
; PILOT; OTH_UNKN                                          35
; OTH_UNKN; PILOT                               

In [28]:
# JW NOTE: Modified the dictionaries below but commenting out for now

In [29]:
# # Pilot
# category_map = {    
#     'CAPTAIN': 'PILOT',
#     'CHECK PILOT': 'PILOT',
#     'CIVL AIR PTRL CHECK PILOT': 'PILOT',
#     'COMMERCIAL STUDENT': 'PILOT',
#     'COMMERCIAL STUDENT PILOT': 'PILOT',
#     'CPL STUDENT': 'PILOT',
#     'DEADHEADING PILOT': 'PILOT',
#     'DRONE OPERATOR': 'PILOT',
#     'FIRST OFFICER': 'PILOT',
#     'HELIPORT MANAGER & PILOT': 'PILOT',
#     'INSTRUCTOR': 'PILOT',
#     'MULTI-ENGINE STUDENT': 'PILOT',
#     'PARAMOTOR PILOT': 'PILOT',
#     'PERSON MANIPULATING CONTROLS (UAS)': 'PILOT',
#     'PIC': 'PILOT',
#     'PIC DURING TIME BUILD': 'PILOT',
#     'PIC NOT FLYING': 'PILOT',
#     'PIC ON A CHECKRIDE': 'PILOT',
#     'PILOT FLYING': 'PILOT',
#     'PILOT IN FOGGLES': 'PILOT',
#     'PILOT NOT FLYING': 'PILOT',
#     'PILOT RECIVING TYPE TRN': 'PILOT',
#     'PRIV. PILOT W/ INSTRUCTOR': 'PILOT',
#     'PRIVATE PILOT': 'PILOT',
#     'RELIEF PILOT': 'PILOT',
#     'REMOTE PIC (UAS)': 'PILOT',
#     'REMOTE PILOT': 'PILOT',
#     'SINGLE PILOT': 'PILOT',
#     'SOLO STUDENT': 'PILOT',
#     'STUDENT': 'PILOT',
#     'STUDENT PILOT': 'PILOT',
#     'STUDENT PILOT & INSTR': 'PILOT',
#     'STUDENT PILOT W/ CFI': 'PILOT',
#     'STUDENT SOLO PILOT': 'PILOT',
#     'TRAINEE': 'PILOT',
#     'UAS SUPERVISOR': 'PILOT',
#     'VISUAL OBSERVER (UAS)': 'PILOT'
# }

In [30]:
# # ATC
# category_map.update ({    
#     'APPROACH': 'ATC',
#     'ATC': 'ATC',
#     'COORDINATOR': 'ATC',
#     'DEPARTURE': 'ATC',
#     'DISPATCHER': 'ATC',
#     'ENROUTE': 'ATC',
#     'FLIGHT DATA / CLEARANCE DELIVERY': 'ATC',
#     'HANDOFF / ASSIST': 'ATC',
#     'SUPERVISOR / CIC': 'ATC',
#     'TRAFFIC MANAGEMENT': 'ATC',
#     'UNICOM OPERATIONS': 'ATC'
# })

In [31]:
# # Ground Operations
# category_map.update ({
#     'AIRPORT MANAGER': 'GROUND',
#     'AIRPORT PERSONNEL': 'GROUND',
#     'FBO MANAGER': 'GROUND',
#     'FBO PERSONNEL': 'GROUND',
#     'GATE AGENT / CSR': 'GROUND',
#     'GROUND': 'GROUND',
#     'JET BRIDGE OPERATOR': 'GROUND',
#     'LOAD CCONTROLLER': 'GROUND',
#     'LOAD PLANNER': 'GROUND',
#     'RAMP': 'GROUND',
#     'VEHICLE DRIVER': 'GROUND'
# })

In [32]:
# # Instruction / Exam
# category_map.update ({
#     'CFI': 'INSTRUCTION / EXAM',
#     'CFI-GFRONT;COMM-GREAR': 'INSTRUCTION / EXAM',
#     'CFII WAS PRESENT': 'INSTRUCTION / EXAM',
#     'DESIGNATED PILOT EXAMINER': 'INSTRUCTION / EXAM',
#     'DPE': 'INSTRUCTION / EXAM',
#     'DUAL FLIGHT': 'INSTRUCTION / EXAM',
#     'DUAL FLIGHT INSTRUCTION': 'INSTRUCTION / EXAM',
#     'DUAL INSTRUCTION FLIGHT': 'INSTRUCTION / EXAM',
#     'FLYING WITH INSTRUCTOR': 'INSTRUCTION / EXAM',
#     'INSTRUMENT STUDENT': 'INSTRUCTION / EXAM',
#     'LINE CHECK AIRMAN': 'INSTRUCTION / EXAM',
#     'LINE CHECK AIRMAN/CAPTAIN': 'INSTRUCTION / EXAM',
#     'LINEMAN': 'INSTRUCTION / EXAM',
#     'SAFETY PILOT': 'INSTRUCTION / EXAM',
#     'STUDENTS INSTRUCTOR': 'INSTRUCTION / EXAM',
#     'SUPERVISOR OF FLYING': 'INSTRUCTION / EXAM',
#     'WITH DPE': 'INSTRUCTION / EXAM',
# })

In [33]:
# # Maintenance / Safety
# category_map.update ({
#     'AVIATION SAFETY OFFICER': 'MAINTENANCE / SAFETY',
#     'FLIGHT ENGINEER / SECOND OFFICER': 'MAINTENANCE / SAFETY',
#     'FLIGHT SAFETY OFFICER': 'MAINTENANCE / SAFETY',
#     'FSDO POI': 'MAINTENANCE / SAFETY',
#     'INSPECTOR': 'MAINTENANCE / SAFETY',
#     'LEAD TECHNICIAN': 'MAINTENANCE / SAFETY',
#     'MAINTENANCE MANAGER': 'MAINTENANCE / SAFETY',
#     'MANAGER': 'MAINTENANCE / SAFETY',
#     'MANAGER - REGULATORY COMPLIANCE': 'MAINTENANCE / SAFETY',
#     'PARTS / STORES PERSONNEL': 'MAINTENANCE / SAFETY',
#     'QUALITY ASSURANCE / AUDIT': 'MAINTENANCE / SAFETY',
#     'SAFETY ANALYST': 'MAINTENANCE / SAFETY',
#     'SAFETY ANANLYST': 'MAINTENANCE / SAFETY',
#     'SAFETY INVESTIGATOR': 'MAINTENANCE / SAFETY',
#     'SAFETY MANAGER': 'MAINTENANCE / SAFETY',
#     'SAFETY OFFICE': 'MAINTENANCE / SAFETY',
#     'SAFETY OFFICER': 'MAINTENANCE / SAFETY',
#     'SMS ANALYST': 'MAINTENANCE / SAFETY',
#     'TECHNICIAN': 'MAINTENANCE / SAFETY'
# })

In [34]:
# # Other Crew
# category_map.update ({
#     'ARMY H-60 CREW': 'OTHER CREW',
#     'FLIGHT ATTENDANT (ON DUTY)': 'OTHER CREW',
#     'FLIGHT ATTENDANT IN CHARGE': 'OTHER CREW',
#     'FLIGHT SERVICE': 'OTHER CREW',
#     'MEDICAL CREW': 'OTHER CREW',
#     'OFF DUTY': 'OTHER CREW'
# })

In [35]:
# # Other / Unknown
# category_map.update ({
#     'CITIZEN': 'OTHER / UNKNOWN',
#     'COMPANY CEO': 'OTHER / UNKNOWN',
#     'LEFT SEAT DURING SIM IFR': 'OTHER / UNKNOWN',
#     'LOCAL': 'OTHER / UNKNOWN',
#     'NO PASSENGERS': 'OTHER / UNKNOWN',
#     'OBSERVER': 'OTHER / UNKNOWN',
#     'OBSERVER/DEADHEADER': 'OTHER / UNKNOWN',
#     'OCEANIC': 'OTHER / UNKNOWN',
#     'OTHER': 'OTHER / UNKNOWN',
#     'OTHER / UNKNOWN': 'OTHER / UNKNOWN',
#     'PASSENGER': 'OTHER / UNKNOWN',
#     'SKYDIVER': 'OTHER / UNKNOWN',
#     'UNKNOWN': 'OTHER / UNKNOWN'
# })

In [36]:
# df_map = df.copy()

In [37]:
# category_map = {
#     # Pilots
#     'Captain': 'Pilots',
#     'First Officer': 'Pilots',
#     'Pilot Flying': 'Pilots',
#     'Pilot Not Flying': 'Pilots',
#     'Single Pilot': 'Pilots',
#     'Relief Pilot': 'Pilots',
#     'Check Pilot': 'Pilots',
#     'Student Pilot': 'Pilots',
#     'Deadheading Pilot': 'Pilots',
#     'Trainee': 'Pilots',
#     'Flight Engineer / Second Officer': 'Pilots',
#     'Drone Operator': 'Pilots',
#     'Remote PIC (UAS)': 'Pilots',
#     'Person Manipulating Controls (UAS)': 'Pilots',
#     'Visual Observer (UAS)': 'Pilots',
#     'remote pilot': 'Pilots',

#     # ATC
#     'Approach': 'ATC',
#     'Enroute': 'ATC',
#     'Local': 'ATC',
#     'Departure': 'ATC',
#     'Oceanic': 'ATC',
#     'Flight Data / Clearance Delivery': 'ATC',
#     'Handoff / Assist': 'ATC',
#     'Traffic Management': 'ATC',
#     'Supervisor / CIC': 'ATC',
#     'Coordinator': 'ATC',

#     # Maintenance / Safety
#     'Technician': 'Maintenance / Safety',
#     'Lead Technician': 'Maintenance / Safety',
#     'Inspector': 'Maintenance / Safety',
#     'Parts / Stores Personnel': 'Maintenance / Safety',
#     'Safety Officer': 'Maintenance / Safety',
#     'Flight Safety Officer': 'Maintenance / Safety',
#     'Safety Manager': 'Maintenance / Safety',
#     'SMS Analyst': 'Maintenance / Safety',
#     'Safety Analyst': 'Maintenance / Safety',
#     'Safety analyst': 'Maintenance / Safety',
#     'Quality Assurance / Audit': 'Maintenance / Safety',
#     'Safety Investigator': 'Maintenance / Safety',
#     'Aviation Safety Officer': 'Maintenance / Safety',

#     # Ground Operations
#     'Ground': 'Ground Operations',
#     'Ramp': 'Ground Operations',
#     'Gate Agent / CSR': 'Ground Operations',
#     'Jet bridge Operator': 'Ground Operations',
#     'Vehicle Driver': 'Ground Operations',
#     'Load Planner': 'Ground Operations',
#     'FBO Personnel': 'Ground Operations',
#     'Airport Personnel': 'Ground Operations',
#     'Airport Manager': 'Ground Operations',

#     # Other Crew
#     'Flight Attendant (On Duty)': 'Other Crew',
#     'Flight Attendant In Charge': 'Other Crew',
#     'Dispatcher': 'Other Crew',
#     'Instructor': 'Other Crew',
#     'Observer': 'Other Crew',
#     'observer': 'Other Crew',
#     'Off Duty': 'Other Crew',
#     'Flight Service': 'Other Crew',
#     'Medical Crew': 'Other Crew',

#     # Unknown
#     'Other / Unknown': 'Unknown',
#     'Other': 'Unknown',
#     'Unknown': 'Unknown',
#     'Passenger': 'Unknown',
#     'Citizen': 'Unknown',
#     'skydiver': 'Unknown',
# }

# df['reporter_function_category'] = df['reporter_function'].map(category_map).fillna('Unknown')

# df.to_csv('Intermediate Input Files/reporter_function.csv', index=False)

### 3.5. Create Dummy Columns

In [38]:
# Obtain dummy columns
df_dummy = df_map.copy()
df_dummy['reporter_function_agg'] = df_dummy['reporter_function_agg'].str.replace(' ', '', regex=False)
dummies = df_dummy['reporter_function_agg'].str.get_dummies(sep=';')

In [39]:
dummies

,ATC,GROUND,INSTRUCTION_EXAM,MAINT_SAFETY,OTH_CREW,OTH_UNKN,PILOT
0,1,0,0,0,0,0,0
1,0,0,0,0,0,0,1
2,0,0,0,0,0,0,1
3,0,0,0,0,0,0,1
4,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...
33718,0,0,0,0,0,0,1
33719,0,0,0,0,0,0,1
33720,0,0,0,0,0,0,1
33721,0,1,0,0,0,0,0


In [40]:
# Rename variables to include prefix "reporter_"
rename = dummies.columns
rename_to = []
for x in rename:
    rename_to.append("reporter_" + x)

dummies.columns=rename_to

In [41]:
dummies

,reporter_ATC,reporter_GROUND,reporter_INSTRUCTION_EXAM,reporter_MAINT_SAFETY,reporter_OTH_CREW,reporter_OTH_UNKN,reporter_PILOT
0,1,0,0,0,0,0,0
1,0,0,0,0,0,0,1
2,0,0,0,0,0,0,1
3,0,0,0,0,0,0,1
4,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...
33718,0,0,0,0,0,0,1
33719,0,0,0,0,0,0,1
33720,0,0,0,0,0,0,1
33721,0,1,0,0,0,0,0


In [42]:
# Old encoding below

In [43]:
# df = pd.read_csv('Int/reporter_function.csv')

# # One-hot encode the category column
# dummies = pd.get_dummies(df['reporter_function_category'], prefix='reporter').astype(int)

# # Insert after reporter_function_category column
# loc_idx = df.columns.get_loc('reporter_function_category') + 1
# for col in reversed(dummies.columns):
#     df.insert(loc_idx, col, dummies[col])

In [44]:
# Concat original/initial-cleaned data with new dummies
data_enc = pd.concat([df.acn, df.reporter_function, dummies], axis=1)

In [45]:
data_enc

,acn,reporter_function,reporter_ATC,reporter_GROUND,reporter_INSTRUCTION_EXAM,reporter_MAINT_SAFETY,reporter_OTH_CREW,reporter_OTH_UNKN,reporter_PILOT
0,1507557,APPROACH,1,0,0,0,0,0,0
1,1513720,CAPTAIN; PILOT NOT FLYING,0,0,0,0,0,0,1
2,1513718,PILOT NOT FLYING,0,0,0,0,0,0,1
3,1513706,FIRST OFFICER; PILOT FLYING,0,0,0,0,0,0,1
4,1513663,PILOT FLYING; CAPTAIN,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...
33718,2192919,INSTRUCTOR,0,0,0,0,0,0,1
33719,2192908,SINGLE PILOT; PILOT FLYING,0,0,0,0,0,0,1
33720,2192884,INSTRUCTOR,0,0,0,0,0,0,1
33721,2192873,RAMP,0,1,0,0,0,0,0


## 4. Export Data

In [46]:
# Export
# data_enc.to_csv('Output Files/Data 4 - reporter_function clean.csv', index=False)